# Merger Integration Tests

Тести інтегрованого in-process мерджера. Клонує гілку `feature/internal-merger` (НЕ main),
щоб тестувати ДО мерджу в main + автодеплою.

**Сценарії:**
- 12.1 Без siblings → мерджер НЕ викликається, normal flow
- 12.2 Siblings + current змерджено в найстаріший → `skipped_merged`
- 12.3 Siblings + current закрито під час мерджу → `skipped_closed`
- 12.4 Current найстаріший → siblings згорнуто в нього, бот працює далі
- 12.5 Мерджер raise → fallback `skipped_merge_candidate`
- 12.6 Siblings але немає requester_id → fallback `skipped_merge_candidate`
- 12.7 Мерджер повертає no_action → fallback `skipped_merge_candidate`


In [ ]:
import os, subprocess, sys
REPO_DIR = '/content/bot'
BRANCH   = 'feature/internal-merger'

# Видаляємо стару копію щоб точно взяти потрібну гілку
if os.path.exists(REPO_DIR):
    subprocess.run(['rm','-rf',REPO_DIR], check=True)

subprocess.run(['git','clone','--depth=1','--branch',BRANCH,
    'https://github.com/tamirmcr1434535/automatization_customer_support.git',
    REPO_DIR], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
    '-r',f'{REPO_DIR}/requirements.txt','google-cloud-secret-manager'], check=True)
sys.path.insert(0, REPO_DIR)
print(f'✅ repo ready (branch: {BRANCH})')


In [ ]:
import os, requests
from google.colab import auth
auth.authenticate_user()
from google.cloud import secretmanager
_sm = secretmanager.SecretManagerServiceClient()
GCP_PROJECT = 'powerful-vine-426615-r2'
def get_secret(name):
    path = f'projects/{GCP_PROJECT}/secrets/{name}/versions/latest'
    return _sm.access_secret_version(request={'name':path}).payload.data.decode().strip()

os.environ['ANTHROPIC_API_KEY']   = get_secret('ANTHROPIC_API_KEY')
os.environ['ZENDESK_API_TOKEN']   = get_secret('ZENDESK_API_TOKEN')
os.environ['STRIPE_SECRET_KEY']   = get_secret('stripe')
os.environ['WOO_CONSUMER_KEY']    = get_secret('WOO_CONSUMER_KEY')
os.environ['WOO_CONSUMER_SECRET'] = get_secret('WOO_CONSUMER_SECRET')
os.environ['ZENDESK_SUBDOMAIN']   = 'iqbooster'
os.environ['ZENDESK_EMAIL']       = 'anna@cellon.ai'
os.environ['WOO_SITE_URL']        = 'https://iqbooster.org'
os.environ['SLACK_BOT_TOKEN']     = 'xoxb-fake'
os.environ['SLACK_TARGET_EMAIL']  = 'fake@fake.test'
os.environ['BQ_PROJECT']          = GCP_PROJECT
os.environ['BQ_DATASET']          = 'fake_dataset'
os.environ['BQ_TABLE']            = 'fake_table'
os.environ['DRY_RUN']             = 'true'
os.environ['TEST_MODE']           = 'false'
print('✅ env ready, DRY_RUN=true')


In [ ]:
import sys, logging
from unittest.mock import patch, MagicMock
from contextlib import ExitStack
import copy

for mod in list(sys.modules.keys()):
    if mod in ('main','classifier','reply_generator','zendesk_client',
               'woocommerce_client','stripe_client','slack_client','bq_logger',
               'ticket_merger'):
        del sys.modules[mod]
sys.modules['bq_logger'] = MagicMock()

import main as bot
assert bot.DRY_RUN, 'DRY_RUN must be True!'
print(f'bot loaded  DRY_RUN={bot.DRY_RUN}')
print(f'ticket_merger module available: {hasattr(bot, "ticket_merger")}')

class WriteTracker:
    def __init__(self):
        self.replies      = []
        self.tags_added   = []
        self.tags_removed = []
        self.solved       = False
        self.set_pending  = False
        self.internal_notes = []
        self.slack_calls  = []
    @property
    def reply_count(self): return len(self.replies)

def _make_ticket(tid='99999', subject='Cancel subscription',
                 body='I want to cancel my subscription',
                 email='test@example.com', tags=None, status='open'):
    return {
        'id': tid, 'subject': subject, 'description': body,
        'tags': tags or [], 'status': status,
        'requester': {'email': email, 'name': 'Test User'},
        'custom_fields': [],
    }

print('helpers ready')


In [ ]:
# ====================================================================
# MERGER INTEGRATION TESTS
# ====================================================================
_PASS = 0
_FAIL = 0
_RESULTS = []

def _run_merger_test(
    name, ticket_data, siblings, merger_status,
    refetch_overrides=None, expect_status=None,
    expect_no_reply=True, requester_id=42,
):
    global _PASS, _FAIL
    tracker = WriteTracker()
    tid = ticket_data.get('id', '99999')
    if isinstance(expect_status, str):
        expect_status = [expect_status]

    ticket_data = copy.deepcopy(ticket_data)
    if requester_id is not None:
        ticket_data.setdefault('requester_id', requester_id)

    refetched = copy.deepcopy(ticket_data)
    if refetch_overrides:
        for k, v in refetch_overrides.items():
            if k == 'tags':
                refetched['tags'] = (refetched.get('tags') or []) + list(v)
            else:
                refetched[k] = v

    get_ticket_seq = [copy.deepcopy(ticket_data), refetched]
    def _get_ticket_side(*_a, **_kw):
        return get_ticket_seq.pop(0) if len(get_ticket_seq) > 1 else refetched

    original_test_mode = bot.TEST_MODE
    bot.TEST_MODE = False

    with ExitStack() as stack:
        stack.enter_context(patch.object(bot.zendesk, 'get_ticket', side_effect=_get_ticket_side))
        stack.enter_context(patch.object(bot.zendesk, 'find_active_tickets_for_email',
            return_value=copy.deepcopy(siblings)))

        if merger_status is None and siblings and requester_id:
            merger_mock = MagicMock(side_effect=RuntimeError('simulated merger crash'))
        else:
            merger_mock = MagicMock(return_value=merger_status or {'status': 'no_action'})
        stack.enter_context(patch('main.ticket_merger.merge_user_tickets', merger_mock))

        def _on_reply(t, b): tracker.replies.append(b)
        def _on_add_tag(t, tag): tracker.tags_added.append(tag)
        def _on_note(t, body): tracker.internal_notes.append(body)
        stack.enter_context(patch.object(bot.zendesk, 'post_reply', side_effect=_on_reply))
        stack.enter_context(patch.object(bot.zendesk, 'post_reply_and_set_pending',
            side_effect=lambda t, b: tracker.replies.append(b)))
        stack.enter_context(patch.object(bot.zendesk, 'add_tag', side_effect=_on_add_tag))
        stack.enter_context(patch.object(bot.zendesk, 'remove_tag', side_effect=lambda *a: None))
        stack.enter_context(patch.object(bot.zendesk, 'solve_ticket', side_effect=lambda t: None))
        stack.enter_context(patch.object(bot.zendesk, 'set_open', return_value=None))
        stack.enter_context(patch.object(bot.zendesk, 'add_internal_note', side_effect=_on_note))
        stack.enter_context(patch.object(bot.zendesk, 'count_bot_replies', return_value=0))
        stack.enter_context(patch.object(bot.zendesk, 'get_ticket_tags',
            return_value=ticket_data.get('tags', [])))
        stack.enter_context(patch.object(bot.zendesk, 'last_public_comment_is_from_agent', return_value=False))
        stack.enter_context(patch.object(bot.zendesk, 'get_first_customer_comment', return_value=None))
        stack.enter_context(patch.object(bot.zendesk, 'get_all_customer_comments_text',
            return_value=ticket_data.get('description', '')))
        stack.enter_context(patch.object(bot.zendesk, 'get_last_customer_comment', return_value=None))
        stack.enter_context(patch.object(bot.zendesk, 'was_recently_handled', return_value=False))
        for sm in ['notify_manual_review', 'notify_not_found', 'notify_refund_skip',
                   'notify_error', 'notify_spam_detected']:
            stack.enter_context(patch.object(bot.slack, sm,
                side_effect=lambda *a, _m=sm, **kw: tracker.slack_calls.append(_m) or True))
        stack.enter_context(patch('main.classify_ticket', return_value={
            'intent': 'TRIAL_CANCELLATION', 'language': 'EN',
            'confidence': 0.95, 'chargeback_risk': '',
        }))
        stack.enter_context(patch.object(bot.woo, 'cancel_subscription', return_value={
            'status': 'dry_run', 'cancelled': True, 'subscription_type': 'trial',
            'subscription_id': 999, 'order_count': 1, 'plan': 'IQ Test',
        }))
        stack.enter_context(patch.object(bot.stripe_cli, 'cancel_subscription',
            return_value={'status': 'not_found'}))
        stack.enter_context(patch.object(bot.stripe_cli, 'find_email_by_last4', return_value=None))
        stack.enter_context(patch('main.log_result', return_value=None))

        try:
            result = bot._process(str(tid))
        except Exception as e:
            result = {'status': 'EXCEPTION', 'error': str(e)}

    bot.TEST_MODE = original_test_mode

    errors = []
    got_status = result.get('status', '?')
    if got_status not in expect_status:
        errors.append(f"status: expected {expect_status}, got '{got_status}'")
    if expect_no_reply and tracker.reply_count > 0:
        errors.append(f'expected NO reply but got {tracker.reply_count}')
    should_call_merger = bool(siblings) and bool(requester_id)
    if should_call_merger and merger_mock.call_count == 0:
        errors.append('expected merger to be invoked but it was not')
    if not should_call_merger and merger_mock.call_count > 0:
        errors.append(f'expected merger NOT invoked but it ran {merger_mock.call_count}x')

    passed = len(errors) == 0
    if passed: _PASS += 1
    else: _FAIL += 1
    _RESULTS.append({'name': name, 'passed': passed, 'errors': errors})
    return passed, errors


print('=' * 74)
print('MERGER INTEGRATION TESTS')
print('=' * 74)

print('--- 12. Merger Integration ---')

_run_merger_test(
    name='12.1 No siblings -> merger NOT called, normal flow proceeds',
    ticket_data=_make_ticket(),
    siblings=[],
    merger_status=None,
    expect_status=['success', 'skipped_not_handled', 'awaiting_card_digits',
                   'manual_review_required'],
    expect_no_reply=False)

_run_merger_test(
    name='12.2 Siblings exist, current merged INTO oldest -> skipped_merged',
    ticket_data=_make_ticket(tid='200'),
    siblings=[{'id': 199, 'subject': 'Older ticket', 'status': 'open'}],
    merger_status={'status': 'merged', 'target_id': 199,
                   'merged_ids': [200], 'current_was_target': False},
    refetch_overrides={'tags': ['merge']},
    expect_status='skipped_merged')

_run_merger_test(
    name='12.3 Siblings + current closed during merge -> skipped_closed',
    ticket_data=_make_ticket(tid='201'),
    siblings=[{'id': 198, 'subject': 'Older', 'status': 'open'}],
    merger_status={'status': 'merged', 'target_id': 198, 'merged_ids': [201]},
    refetch_overrides={'status': 'closed'},
    expect_status='skipped_closed')

_run_merger_test(
    name='12.4 Current IS oldest -> siblings folded INTO current, bot continues',
    ticket_data=_make_ticket(tid='150'),
    siblings=[{'id': 151, 'subject': 'Newer dup', 'status': 'new'}],
    merger_status={'status': 'merged', 'target_id': 150,
                   'merged_ids': [151], 'current_was_target': True},
    refetch_overrides=None,
    expect_status=['success', 'skipped_not_handled', 'awaiting_card_digits',
                   'manual_review_required'],
    expect_no_reply=False)

_run_merger_test(
    name='12.5 Merger raises -> falls through to skipped_merge_candidate',
    ticket_data=_make_ticket(tid='250'),
    siblings=[{'id': 249, 'subject': 'Older', 'status': 'open'}],
    merger_status=None,
    refetch_overrides=None,
    expect_status='skipped_merge_candidate')

_run_merger_test(
    name='12.6 Siblings but no requester_id -> merger NOT called, fallback',
    ticket_data=_make_ticket(tid='260'),
    siblings=[{'id': 259, 'subject': 'Older', 'status': 'open'}],
    merger_status=None,
    refetch_overrides=None,
    expect_status='skipped_merge_candidate',
    requester_id=None)

_run_merger_test(
    name='12.7 Merger returns no_action -> falls through to skipped_merge_candidate',
    ticket_data=_make_ticket(tid='270'),
    siblings=[{'id': 269, 'subject': 'Older', 'status': 'open'}],
    merger_status={'status': 'no_action', 'active_count': 1},
    refetch_overrides=None,
    expect_status='skipped_merge_candidate')

print()
print('=' * 74)
print(f'RESULTS: {_PASS} passed, {_FAIL} failed out of {_PASS + _FAIL} tests')
print('=' * 74)
for r in _RESULTS:
    icon = 'PASS' if r['passed'] else 'FAIL'
    print(f"  [{icon}] {r['name']}")
    if not r['passed']:
        for err in r['errors']:
            print(f'         {err}')
print()
if _FAIL > 0:
    print(f'DEPLOYMENT BLOCKED: {_FAIL} merger test(s) failed.')
else:
    print('ALL MERGER TESTS PASSED. Safe to merge to main.')
